<a href="https://colab.research.google.com/github/HectorArielBaez/RegresionAvanzada/blob/main/RL_Expicativo_de_importancia_de_variables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # selecciona kaggle.json desde tu PC

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"arielbaez","key":"bd5916039ad5e50e7ae33fd4af856b10"}'}

In [ ]:
# 1. Instalar Kaggle y subir la API token
!pip install kaggle

# Luego, subí tu archivo `kaggle.json` en el directorio raíz de Colab

# 2. Descargar dataset de Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Por ejemplo, con el dataset 'harunshimanto/epileptic-seizure-recognition'
!kaggle datasets download -d harunshimanto/epileptic-seizure-recognition

# 3. Descomprimir
!unzip -o epileptic-seizure-recognition.zip

Dataset URL: https://www.kaggle.com/datasets/harunshimanto/epileptic-seizure-recognition
License(s): other
  0% 0.00/2.77M [00:00<?, ?B/s]
100% 2.77M/2.77M [00:00<00:00, 656MB/s]
Archive:  epileptic-seizure-recognition.zip
  inflating: Epileptic Seizure Recognition.csv  


In [ ]:
# ===============================================
# 2. Librerías y carga de datos
# ===============================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [ ]:
# Cargar dataset
df = pd.read_csv("/content/Epileptic Seizure Recognition.csv")

In [ ]:
# Target: columna 'y' (1 = epilepsia, 0 = no epilepsia)
df['y'] = df['y'].replace(5, 0)

X = df.drop(columns=['y'])
y = df['y']

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Escalado para regresión logística
scaler = StandardScaler()

# Drop the 'Unnamed' column before scaling
X_train_scaled = scaler.fit_transform(X_train.drop(columns=['Unnamed']))
X_test_scaled = scaler.transform(X_test.drop(columns=['Unnamed']))

In [ ]:
# ===============================================
# 3. Modelos
# ===============================================

# --- Regresión Logística ---
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(X_train.drop(columns=['Unnamed']), y_train)
y_pred_rf = rf.predict(X_test.drop(columns=['Unnamed']))
y_proba_rf = rf.predict_proba(X_test.drop(columns=['Unnamed']))

# --- XGBoost ---
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train.drop(columns=['Unnamed']), y_train)
y_pred_xgb = xgb.predict(X_test.drop(columns=['Unnamed']))
y_proba_xgb = xgb.predict_proba(X_test.drop(columns=['Unnamed']))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [16:15:36] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# ===============================================
# 4. Comparación de resultados
# ===============================================
def get_metrics(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba, multi_class='ovr'),
        "Recall (clase 1)": classification_report(y_true, y_pred, output_dict=True)["1"]["recall"],
        "F1-score (clase 1)": classification_report(y_true, y_pred, output_dict=True)["1"]["f1-score"]
    }

results = pd.DataFrame({
    "Logistic Regression": get_metrics(y_test, y_pred_lr, y_proba_lr),
    "Random Forest": get_metrics(y_test, y_pred_rf, y_proba_rf),
    "XGBoost": get_metrics(y_test, y_pred_xgb, y_proba_xgb)
}).T

print("📊 Comparación de modelos")
print(results)

📊 Comparación de modelos
                     Accuracy   ROC-AUC  Recall (clase 1)  F1-score (clase 1)
Logistic Regression  0.251739  0.538043          0.360870            0.386496
Random Forest        0.701304  0.920502          0.947826            0.929638
XGBoost              0.688261  0.914970          0.923913            0.938190


In [ ]:
# ===============================================
# 5. Interpretación de variables
# ===============================================

# --- Coeficientes RL ---
coef_df = pd.DataFrame({
    "Variable": X.drop(columns=['Unnamed']).columns,
    "Coeficiente": log_reg.coef_[0]
}).sort_values(by="Coeficiente", ascending=False)

print("\n🔹 Variables más influyentes según RL:")
display(coef_df.head())

# --- Importancia en RF ---
importances_rf = pd.DataFrame({
    "Variable": X.drop(columns=['Unnamed']).columns,
    "Importancia": rf.feature_importances_
}).sort_values(by="Importancia", ascending=False)

print("\n🌳 Variables más importantes según RF:")
display(importances_rf.head())

# --- Importancia en XGB ---
importances_xgb = pd.DataFrame({
    "Variable": X.drop(columns=['Unnamed']).columns,
    "Importancia": xgb.feature_importances_
}).sort_values(by="Importancia", ascending=False)

print("\n⚡ Variables más importantes según XGB:")
display(importances_xgb.head())


🔹 Variables más influyentes según RL:


,Variable,Coeficiente
124,X125,0.745274
127,X128,0.648366
39,X40,0.553461
31,X32,0.461201
85,X86,0.455269



🌳 Variables más importantes según RF:


,Variable,Importancia
160,X161,0.009516
12,X13,0.009084
43,X44,0.008593
36,X37,0.008167
155,X156,0.008091



⚡ Variables más importantes según XGB:


,Variable,Importancia
158,X159,0.043802
43,X44,0.016290
92,X93,0.015632
42,X43,0.013360
107,X108,0.012094
